In [6]:
# %%
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, classification_report

# ---- Paths ----
base_path = r"C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing"
data_path = os.path.join(base_path, "65_Nutrients_Data.csv")  # change to 102_ or FDA_ as needed
save_path = os.path.join(base_path, "Baseline_Models_65_Nutrients")
os.makedirs(save_path, exist_ok=True)

# ---- Load Data ----
data = pd.read_csv(data_path, index_col=0)
X = data.drop(columns=['novaclass'])
y = data['novaclass']

# ---- Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Scaling ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---- Helper Function ----
def evaluate_and_save(model, model_name, X_train, X_test, scaled=False, xgb_fix=False):
    # For XGBoost only: shift classes to start from 0
    if xgb_fix:
        y_train_fixed = y_train - 1
        y_test_fixed = y_test - 1
        model.fit(X_train_scaled if scaled else X_train, y_train_fixed)
        y_pred = model.predict(X_test_scaled if scaled else X_test)
        acc = accuracy_score(y_test_fixed, y_pred)
        f1 = f1_score(y_test_fixed, y_pred, average='weighted')
        prec = precision_score(y_test_fixed, y_pred, average='weighted')
        rec = recall_score(y_test_fixed, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_test_fixed, y_pred)
    else:
        model.fit(X_train_scaled if scaled else X_train, y_train)
        y_pred = model.predict(X_test_scaled if scaled else X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_test, y_pred)
    
    print(f"\n🧠 {model_name} Results:")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, MCC: {mcc:.4f}")
    print(classification_report(y_test, y_pred))
    joblib.dump(model, os.path.join(save_path, f"{model_name}_baseline.pkl"))
    return [model_name, f1, mcc, acc, prec, rec]

# %%
results = []

# ---- 1. LightGBM ----
from lightgbm import LGBMClassifier
results.append(evaluate_and_save(LGBMClassifier(random_state=42), "LightGBM", X_train, X_test, scaled=True))

# ---- 2. Gradient Boosting ----
from sklearn.ensemble import GradientBoostingClassifier
results.append(evaluate_and_save(GradientBoostingClassifier(random_state=42), "GradientBoost", X_train, X_test, scaled=True))

# ---- 3. Random Forest ----
from sklearn.ensemble import RandomForestClassifier
results.append(evaluate_and_save(RandomForestClassifier(random_state=42), "RandomForest", X_train, X_test))

# ---- 4. XGBoost (with class label fix) ----
from xgboost import XGBClassifier
results.append(evaluate_and_save(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    "XGBoost",
    X_train,
    X_test,
    scaled=True,
    xgb_fix=True
))

# ---- 5. Extra Trees ----
from sklearn.ensemble import ExtraTreesClassifier
results.append(evaluate_and_save(ExtraTreesClassifier(random_state=42), "ExtraTrees", X_train, X_test))

# ---- 6. MLP ----
from sklearn.neural_network import MLPClassifier
results.append(evaluate_and_save(MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42), "MLP", X_train, X_test, scaled=True))

# ---- 7. KNN ----
from sklearn.neighbors import KNeighborsClassifier
results.append(evaluate_and_save(KNeighborsClassifier(n_neighbors=5), "KNN", X_train, X_test, scaled=True))

# ---- 8. Decision Tree ----
from sklearn.tree import DecisionTreeClassifier
results.append(evaluate_and_save(DecisionTreeClassifier(random_state=42), "DecisionTree", X_train, X_test))

# ---- 9. Logistic Regression ----
from sklearn.linear_model import LogisticRegression
results.append(evaluate_and_save(LogisticRegression(max_iter=1000, random_state=42), "LogisticRegression", X_train, X_test, scaled=True))

# ---- 10. SVM ----
from sklearn.svm import SVC
results.append(evaluate_and_save(SVC(kernel='rbf', random_state=42), "SVM", X_train, X_test, scaled=True))

# ---- 11. AdaBoost ----
from sklearn.ensemble import AdaBoostClassifier
results.append(evaluate_and_save(AdaBoostClassifier(random_state=42), "AdaBoost", X_train, X_test))

# ---- 12. Naive Bayes ----
from sklearn.naive_bayes import GaussianNB
results.append(evaluate_and_save(GaussianNB(), "NaiveBayes", X_train, X_test, scaled=True))

# %%
# ---- Save Summary ----
results_df = pd.DataFrame(results, columns=["Model", "F1 Score", "MCC", "Accuracy", "Precision", "Recall"])
results_df.to_csv(os.path.join(save_path, "baseline_results_summary.csv"), index=False)
print("\n✅ All 12 baseline models trained and saved successfully in:")
print(save_path)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000903 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11562
[LightGBM] [Info] Number of data points in the train set: 2376, number of used features: 64
[LightGBM] [Info] Start training from score -2.171055
[LightGBM] [Info] Start training from score -4.035504
[LightGBM] [Info] Start training from score -1.851595
[LightGBM] [Info] Start training from score -0.340690
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



🧠 GradientBoost Results:
Accuracy: 0.9040, F1: 0.9014, Precision: 0.9012, Recall: 0.9040, MCC: 0.7844
              precision    recall  f1-score   support

           1       0.85      0.76      0.81        68
           2       0.69      0.82      0.75        11
           3       0.80      0.70      0.75        93
           4       0.94      0.97      0.95       422

    accuracy                           0.90       594
   macro avg       0.82      0.81      0.81       594
weighted avg       0.90      0.90      0.90       594


🧠 RandomForest Results:
Accuracy: 0.9125, F1: 0.9095, Precision: 0.9088, Recall: 0.9125, MCC: 0.8027
              precision    recall  f1-score   support

           1       0.84      0.76      0.80        68
           2       0.90      0.82      0.86        11
           3       0.81      0.71      0.76        93
           4       0.94      0.98      0.96       422

    accuracy                           0.91       594
   macro avg       0.87      0.82 

C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [07:58:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🧠 XGBoost Results:
Accuracy: 0.9259, F1: 0.9246, Precision: 0.9248, Recall: 0.9259, MCC: 0.8358
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        68
           2       0.00      0.00      0.00        11
           3       0.03      0.15      0.05        93
           4       0.00      0.00      0.00       422

    accuracy                           0.02       594
   macro avg       0.01      0.03      0.01       594
weighted avg       0.01      0.02      0.01       594



C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificati


🧠 ExtraTrees Results:
Accuracy: 0.9024, F1: 0.8983, Precision: 0.8983, Recall: 0.9024, MCC: 0.7788
              precision    recall  f1-score   support

           1       0.78      0.72      0.75        68
           2       0.90      0.82      0.86        11
           3       0.84      0.68      0.75        93
           4       0.93      0.98      0.96       422

    accuracy                           0.90       594
   macro avg       0.86      0.80      0.83       594
weighted avg       0.90      0.90      0.90       594


🧠 MLP Results:
Accuracy: 0.8822, F1: 0.8821, Precision: 0.8821, Recall: 0.8822, MCC: 0.7422
              precision    recall  f1-score   support

           1       0.72      0.72      0.72        68
           2       0.70      0.64      0.67        11
           3       0.72      0.73      0.73        93
           4       0.95      0.95      0.95       422

    accuracy                           0.88       594
   macro avg       0.77      0.76      0.77   

In [7]:
# %%
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, classification_report

# ---- Paths ----
base_path = r"C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing"
data_path = os.path.join(base_path, "102_Nutrients_Data.csv")  # change to 102_ or FDA_ as needed
save_path = os.path.join(base_path, "Baseline_Models_102_Nutrients")
os.makedirs(save_path, exist_ok=True)

# ---- Load Data ----
data = pd.read_csv(data_path, index_col=0)
X = data.drop(columns=['novaclass'])
y = data['novaclass']

# ---- Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Scaling ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---- Helper Function ----
def evaluate_and_save(model, model_name, X_train, X_test, scaled=False, xgb_fix=False):
    # For XGBoost only: shift classes to start from 0
    if xgb_fix:
        y_train_fixed = y_train - 1
        y_test_fixed = y_test - 1
        model.fit(X_train_scaled if scaled else X_train, y_train_fixed)
        y_pred = model.predict(X_test_scaled if scaled else X_test)
        acc = accuracy_score(y_test_fixed, y_pred)
        f1 = f1_score(y_test_fixed, y_pred, average='weighted')
        prec = precision_score(y_test_fixed, y_pred, average='weighted')
        rec = recall_score(y_test_fixed, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_test_fixed, y_pred)
    else:
        model.fit(X_train_scaled if scaled else X_train, y_train)
        y_pred = model.predict(X_test_scaled if scaled else X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_test, y_pred)
    
    print(f"\n🧠 {model_name} Results:")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, MCC: {mcc:.4f}")
    print(classification_report(y_test, y_pred))
    joblib.dump(model, os.path.join(save_path, f"{model_name}_baseline.pkl"))
    return [model_name, f1, mcc, acc, prec, rec]

# %%
results = []

# ---- 1. LightGBM ----
from lightgbm import LGBMClassifier
results.append(evaluate_and_save(LGBMClassifier(random_state=42), "LightGBM", X_train, X_test, scaled=True))

# ---- 2. Gradient Boosting ----
from sklearn.ensemble import GradientBoostingClassifier
results.append(evaluate_and_save(GradientBoostingClassifier(random_state=42), "GradientBoost", X_train, X_test, scaled=True))

# ---- 3. Random Forest ----
from sklearn.ensemble import RandomForestClassifier
results.append(evaluate_and_save(RandomForestClassifier(random_state=42), "RandomForest", X_train, X_test))

# ---- 4. XGBoost (with class label fix) ----
from xgboost import XGBClassifier
results.append(evaluate_and_save(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    "XGBoost",
    X_train,
    X_test,
    scaled=True,
    xgb_fix=True
))

# ---- 5. Extra Trees ----
from sklearn.ensemble import ExtraTreesClassifier
results.append(evaluate_and_save(ExtraTreesClassifier(random_state=42), "ExtraTrees", X_train, X_test))

# ---- 6. MLP ----
from sklearn.neural_network import MLPClassifier
results.append(evaluate_and_save(MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42), "MLP", X_train, X_test, scaled=True))

# ---- 7. KNN ----
from sklearn.neighbors import KNeighborsClassifier
results.append(evaluate_and_save(KNeighborsClassifier(n_neighbors=5), "KNN", X_train, X_test, scaled=True))

# ---- 8. Decision Tree ----
from sklearn.tree import DecisionTreeClassifier
results.append(evaluate_and_save(DecisionTreeClassifier(random_state=42), "DecisionTree", X_train, X_test))

# ---- 9. Logistic Regression ----
from sklearn.linear_model import LogisticRegression
results.append(evaluate_and_save(LogisticRegression(max_iter=1000, random_state=42), "LogisticRegression", X_train, X_test, scaled=True))

# ---- 10. SVM ----
from sklearn.svm import SVC
results.append(evaluate_and_save(SVC(kernel='rbf', random_state=42), "SVM", X_train, X_test, scaled=True))

# ---- 11. AdaBoost ----
from sklearn.ensemble import AdaBoostClassifier
results.append(evaluate_and_save(AdaBoostClassifier(random_state=42), "AdaBoost", X_train, X_test))

# ---- 12. Naive Bayes ----
from sklearn.naive_bayes import GaussianNB
results.append(evaluate_and_save(GaussianNB(), "NaiveBayes", X_train, X_test, scaled=True))

# %%
# ---- Save Summary ----
results_df = pd.DataFrame(results, columns=["Model", "F1 Score", "MCC", "Accuracy", "Precision", "Recall"])
results_df.to_csv(os.path.join(save_path, "baseline_results_summary.csv"), index=False)
print("\n✅ All 12 baseline models trained and saved successfully in:")
print(save_path)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001774 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13826
[LightGBM] [Info] Number of data points in the train set: 2376, number of used features: 96
[LightGBM] [Info] Start training from score -2.171055
[LightGBM] [Info] Start training from score -4.035504
[LightGBM] [Info] Start training from score -1.851595
[LightGBM] [Info] Start training from score -0.340690
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



🧠 GradientBoost Results:
Accuracy: 0.9040, F1: 0.9014, Precision: 0.9018, Recall: 0.9040, MCC: 0.7832
              precision    recall  f1-score   support

           1       0.91      0.75      0.82        68
           2       0.82      0.82      0.82        11
           3       0.79      0.72      0.75        93
           4       0.93      0.97      0.95       422

    accuracy                           0.90       594
   macro avg       0.86      0.82      0.84       594
weighted avg       0.90      0.90      0.90       594


🧠 RandomForest Results:
Accuracy: 0.9007, F1: 0.8970, Precision: 0.8962, Recall: 0.9007, MCC: 0.7758
              precision    recall  f1-score   support

           1       0.77      0.74      0.75        68
           2       0.90      0.82      0.86        11
           3       0.81      0.67      0.73        93
           4       0.94      0.98      0.96       422

    accuracy                           0.90       594
   macro avg       0.85      0.80 

C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [07:59:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🧠 XGBoost Results:
Accuracy: 0.9259, F1: 0.9242, Precision: 0.9238, Recall: 0.9259, MCC: 0.8349
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        68
           2       0.00      0.00      0.00        11
           3       0.03      0.16      0.06        93
           4       0.00      0.00      0.00       422

    accuracy                           0.03       594
   macro avg       0.01      0.03      0.01       594
weighted avg       0.01      0.03      0.01       594



C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificati


🧠 ExtraTrees Results:
Accuracy: 0.9040, F1: 0.9001, Precision: 0.8995, Recall: 0.9040, MCC: 0.7827
              precision    recall  f1-score   support

           1       0.77      0.69      0.73        68
           2       0.90      0.82      0.86        11
           3       0.83      0.70      0.76        93
           4       0.93      0.99      0.96       422

    accuracy                           0.90       594
   macro avg       0.86      0.80      0.83       594
weighted avg       0.90      0.90      0.90       594


🧠 MLP Results:
Accuracy: 0.8788, F1: 0.8787, Precision: 0.8799, Recall: 0.8788, MCC: 0.7342
              precision    recall  f1-score   support

           1       0.67      0.75      0.71        68
           2       0.89      0.73      0.80        11
           3       0.76      0.70      0.73        93
           4       0.94      0.94      0.94       422

    accuracy                           0.88       594
   macro avg       0.82      0.78      0.79   

In [8]:
# %%
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, classification_report

# ---- Paths ----
base_path = r"C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing"
data_path = os.path.join(base_path, "FDA_Nutrients.csv")  # change to 102_ or FDA_ as needed
save_path = os.path.join(base_path, "Baseline_Models_FDA_Nutrients")
os.makedirs(save_path, exist_ok=True)

# ---- Load Data ----
data = pd.read_csv(data_path, index_col=0)
X = data.drop(columns=['novaclass'])
y = data['novaclass']

# ---- Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Scaling ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---- Helper Function ----
def evaluate_and_save(model, model_name, X_train, X_test, scaled=False, xgb_fix=False):
    # For XGBoost only: shift classes to start from 0
    if xgb_fix:
        y_train_fixed = y_train - 1
        y_test_fixed = y_test - 1
        model.fit(X_train_scaled if scaled else X_train, y_train_fixed)
        y_pred = model.predict(X_test_scaled if scaled else X_test)
        acc = accuracy_score(y_test_fixed, y_pred)
        f1 = f1_score(y_test_fixed, y_pred, average='weighted')
        prec = precision_score(y_test_fixed, y_pred, average='weighted')
        rec = recall_score(y_test_fixed, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_test_fixed, y_pred)
    else:
        model.fit(X_train_scaled if scaled else X_train, y_train)
        y_pred = model.predict(X_test_scaled if scaled else X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        mcc = matthews_corrcoef(y_test, y_pred)
    
    print(f"\n🧠 {model_name} Results:")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, MCC: {mcc:.4f}")
    print(classification_report(y_test, y_pred))
    joblib.dump(model, os.path.join(save_path, f"{model_name}_baseline.pkl"))
    return [model_name, f1, mcc, acc, prec, rec]

# %%
results = []

# ---- 1. LightGBM ----
from lightgbm import LGBMClassifier
results.append(evaluate_and_save(LGBMClassifier(random_state=42), "LightGBM", X_train, X_test, scaled=True))

# ---- 2. Gradient Boosting ----
from sklearn.ensemble import GradientBoostingClassifier
results.append(evaluate_and_save(GradientBoostingClassifier(random_state=42), "GradientBoost", X_train, X_test, scaled=True))

# ---- 3. Random Forest ----
from sklearn.ensemble import RandomForestClassifier
results.append(evaluate_and_save(RandomForestClassifier(random_state=42), "RandomForest", X_train, X_test))

# ---- 4. XGBoost (with class label fix) ----
from xgboost import XGBClassifier
results.append(evaluate_and_save(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    "XGBoost",
    X_train,
    X_test,
    scaled=True,
    xgb_fix=True
))

# ---- 5. Extra Trees ----
from sklearn.ensemble import ExtraTreesClassifier
results.append(evaluate_and_save(ExtraTreesClassifier(random_state=42), "ExtraTrees", X_train, X_test))

# ---- 6. MLP ----
from sklearn.neural_network import MLPClassifier
results.append(evaluate_and_save(MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42), "MLP", X_train, X_test, scaled=True))

# ---- 7. KNN ----
from sklearn.neighbors import KNeighborsClassifier
results.append(evaluate_and_save(KNeighborsClassifier(n_neighbors=5), "KNN", X_train, X_test, scaled=True))

# ---- 8. Decision Tree ----
from sklearn.tree import DecisionTreeClassifier
results.append(evaluate_and_save(DecisionTreeClassifier(random_state=42), "DecisionTree", X_train, X_test))

# ---- 9. Logistic Regression ----
from sklearn.linear_model import LogisticRegression
results.append(evaluate_and_save(LogisticRegression(max_iter=1000, random_state=42), "LogisticRegression", X_train, X_test, scaled=True))

# ---- 10. SVM ----
from sklearn.svm import SVC
results.append(evaluate_and_save(SVC(kernel='rbf', random_state=42), "SVM", X_train, X_test, scaled=True))

# ---- 11. AdaBoost ----
from sklearn.ensemble import AdaBoostClassifier
results.append(evaluate_and_save(AdaBoostClassifier(random_state=42), "AdaBoost", X_train, X_test))

# ---- 12. Naive Bayes ----
from sklearn.naive_bayes import GaussianNB
results.append(evaluate_and_save(GaussianNB(), "NaiveBayes", X_train, X_test, scaled=True))

# %%
# ---- Save Summary ----
results_df = pd.DataFrame(results, columns=["Model", "F1 Score", "MCC", "Accuracy", "Precision", "Recall"])
results_df.to_csv(os.path.join(save_path, "baseline_results_summary.csv"), index=False)
print("\n✅ All 12 baseline models trained and saved successfully in:")
print(save_path)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000168 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2564
[LightGBM] [Info] Number of data points in the train set: 2376, number of used features: 12
[LightGBM] [Info] Start training from score -2.171055
[LightGBM] [Info] Start training from score -4.035504
[LightGBM] [Info] Start training from score -1.851595
[LightGBM] [Info] Start training from score -0.340690
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



🧠 LightGBM Results:
Accuracy: 0.9158, F1: 0.9133, Precision: 0.9129, Recall: 0.9158, MCC: 0.8105
              precision    recall  f1-score   support

           1       0.85      0.76      0.81        68
           2       0.90      0.82      0.86        11
           3       0.83      0.74      0.78        93
           4       0.94      0.98      0.96       422

    accuracy                           0.92       594
   macro avg       0.88      0.83      0.85       594
weighted avg       0.91      0.92      0.91       594


🧠 GradientBoost Results:
Accuracy: 0.8973, F1: 0.8939, Precision: 0.8929, Recall: 0.8973, MCC: 0.7678
              precision    recall  f1-score   support

           1       0.82      0.74      0.78        68
           2       0.90      0.82      0.86        11
           3       0.78      0.68      0.72        93
           4       0.93      0.97      0.95       422

    accuracy                           0.90       594
   macro avg       0.86      0.80     

C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [07:59:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🧠 XGBoost Results:
Accuracy: 0.9091, F1: 0.9064, Precision: 0.9056, Recall: 0.9091, MCC: 0.7956
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        68
           2       0.00      0.00      0.00        11
           3       0.05      0.22      0.08        93
           4       0.00      0.00      0.00       422

    accuracy                           0.03       594
   macro avg       0.01      0.04      0.02       594
weighted avg       0.01      0.03      0.01       594



C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificati


🧠 ExtraTrees Results:
Accuracy: 0.9125, F1: 0.9090, Precision: 0.9090, Recall: 0.9125, MCC: 0.8025
              precision    recall  f1-score   support

           1       0.81      0.75      0.78        68
           2       0.91      0.91      0.91        11
           3       0.86      0.70      0.77        93
           4       0.94      0.99      0.96       422

    accuracy                           0.91       594
   macro avg       0.88      0.84      0.85       594
weighted avg       0.91      0.91      0.91       594



C:\Users\hvish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(



🧠 MLP Results:
Accuracy: 0.8771, F1: 0.8759, Precision: 0.8749, Recall: 0.8771, MCC: 0.7280
              precision    recall  f1-score   support

           1       0.70      0.68      0.69        68
           2       0.82      0.82      0.82        11
           3       0.73      0.70      0.71        93
           4       0.94      0.95      0.94       422

    accuracy                           0.88       594
   macro avg       0.80      0.79      0.79       594
weighted avg       0.87      0.88      0.88       594


🧠 KNN Results:
Accuracy: 0.8704, F1: 0.8718, Precision: 0.8750, Recall: 0.8704, MCC: 0.7187
              precision    recall  f1-score   support

           1       0.62      0.72      0.67        68
           2       1.00      0.73      0.84        11
           3       0.75      0.71      0.73        93
           4       0.94      0.93      0.94       422

    accuracy                           0.87       594
   macro avg       0.83      0.77      0.79       594

In [9]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

base_path = Path(r"C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing")

folders = [
    ("65 nutrients", base_path / "Baseline_Models_65_Nutrients"),
    ("102 nutrients", base_path / "Baseline_Models_102_Nutrients"),
    ("FDA nutrients", base_path / "Baseline_Models_FDA_Nutrients"),
]

plot_out = base_path / "plots"
plot_out.mkdir(exist_ok=True)

all_summaries = []
for name, folder in folders:
    csv_path = folder / "baseline_results_summary.csv"
    if not csv_path.exists():
        print(f"Missing: {csv_path} -- skipping")
        continue

    df = pd.read_csv(csv_path)
    df["dataset"] = name
    all_summaries.append(df)

    # Melt for seaborn
    melt = df.melt(id_vars=["Model"], value_vars=["F1 Score", "MCC", "Accuracy", "Precision", "Recall"],
                   var_name="metric", value_name="value")

    plt.figure(figsize=(12, 6))
    sns.barplot(data=melt, x="Model", y="value", hue="metric")
    plt.title(f"Baseline model metrics — {name}")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1.0)
    plt.ylabel("Score")
    plt.tight_layout()
    out_file = plot_out / f"baseline_metrics_{name.replace(' ', '_')}.png"
    plt.savefig(out_file, dpi=200)
    plt.close()
    print(f"Saved plot: {out_file}")

# Combined comparison (F1 across datasets)
if all_summaries:
    combined = pd.concat(all_summaries, ignore_index=True)
    pivot_f1 = combined.pivot_table(index="Model", columns="dataset", values="F1 Score")
    plt.figure(figsize=(10, max(4, 0.4 * len(pivot_f1))))
    sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="viridis", cbar_kws={"label": "F1 Score"})
    plt.title("F1 Score comparison across datasets")
    plt.tight_layout()
    out_heat = plot_out / "baseline_F1_comparison_heatmap.png"
    plt.savefig(out_heat, dpi=200)
    plt.close()
    print(f"Saved combined F1 heatmap: {out_heat}")

    # Save combined CSV for further analysis
    combined_csv = plot_out / "combined_baseline_results.csv"
    combined.to_csv(combined_csv, index=False)
    print(f"Saved combined CSV: {combined_csv}")
else:
    print("No summary files found in the expected folders.")

Saved plot: C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing\plots\baseline_metrics_65_nutrients.png
Saved plot: C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing\plots\baseline_metrics_102_nutrients.png
Saved plot: C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing\plots\baseline_metrics_FDA_nutrients.png
Saved combined F1 heatmap: C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing\plots\baseline_F1_comparison_heatmap.png
Saved combined CSV: C:\Users\hvish\Desktop\BTP new\Numerical Model Food Processing\plots\combined_baseline_results.csv
